In [ ]:
import random
import torchvision
import cv2
import numpy as np
import torch
torch.set_grad_enabled(False)
import time

model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
# Uses GPU if it can but otherwise uses CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.eval().to(device)
# Set seed so is the same each time
random.seed(14)

def main():
    # Open webcam
    cap = cv2.VideoCapture(0)
    # Loops forever until webcam is closed or application is quit
    while True:
        # Read webcam frame
        result = cap.read()
        # Ret returns true or false depending whether frame was successfully read
        ret = result[0]
        # If frame not successful then break from loop
        if ret is False:
            print("Frame failed to be read")
            break
        frame = result[1]
        t = time.time()
        # Convert the frame to RGB for the PyTorch format, as OpenCV uses BGR
        frameUpdate = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Convert to PyTorch tensor
        image_tensor = torchvision.transforms.functional.to_tensor(frameUpdate).to(device)
        # Run model on image tensor
        output = model([image_tensor])[0]
        print('executed in %.3fs' % (time.time() - t))
        # Copy frame as to not alter the original frame
        result_image2 = frameUpdate.copy()
        # Merge all masks into a single mask for all detected objects
        masks = None
        for score, mask in zip(output['scores'], output['masks']):
            if score > 0.5:
                if masks is None:
                    masks = mask
                else:
                    # Compare two masks and merge with higher value pixel by pixel
                    masks = torch.max(masks, mask)
        # If masks exist then output them onto the image
        if masks is not None:
            # Remove extra dimension, move to CPU and convert to numpy array
            current_mask = masks.squeeze(0).cpu().numpy()
            # Binary mask that turbns pixels with above 0.5 in 1 and all others as 0
            new_mask = (current_mask > 0.5).astype(np.uint8)
            # Create empty image with zeros (same size result image)
            coloured = np.zeros_like(result_image2)
            # Make coloured red (BGR, blue, green, red)
            coloured[:, :] = (0, 0, 255)
            # Turn mask 3D by stacking 3 times of the same mask
            masks_3 = np.stack([new_mask] * 3, axis=-1)
            #
            result_image3 = np.where(masks_3, cv2.addWeighted(result_image2, 0.5, coloured, 0.5, 0), result_image2).astype(np.uint8)
        else:
            # Otherwise use the same image unaltered
            result_image3 = result_image2.copy()
        # Coco labels
        coco_names = ['unlabeled', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'hat', 'backpack', 'umbrella', 'shoe', 'eye glasses', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'plate', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'mirror', 'dining table', 'window', 'desk', 'toilet', 'door', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'blender', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
        # Colors for each label
        colors = [[random.randint(0, 255) for _ in range(3)] for _ in coco_names]
        # Make copy again
        result_image4 = result_image3.copy()

        for box, label, score in zip(output['boxes'], output['labels'], output['scores']):
            if score > 0.5:
                # Take random colour for each box and label
                color = random.choice(colors)
                # Calculate the thickness
                tl = round(0.002 * max(result_image4.shape[0:2])) + 1
                # Take coordinates for box
                c1  = (int(box[0]), int(box[1]))
                c2  = (int(box[2]), int(box[3]))
                # Draw rectangle by using current result image, coords, colour and thickness calculated earlier
                cv2.rectangle(result_image4, c1, c2, color, thickness=tl)
                # Create the text to display the label and its score
                display_txt = "%s: %.1f%%" % (coco_names[label], 100 * score)
                tf = max(tl - 1, 1)
                # Calculate size of current text box
                t_size = cv2.getTextSize(display_txt, 0, fontScale=tl / 3, thickness=tf)[0]
                # Calculate coords for text box
                c2 = c1[0] + t_size[0], c1[1] - t_size[1] - 3
                # Draw coloured rectangle
                cv2.rectangle(result_image4, c1, c2, color, -1)
                # Draw text on top of this rectangle
                cv2.putText(result_image4, display_txt, (c1[0], c1[1] - 2), 0, tl / 3, [225, 255, 255], thickness=tf, lineType=cv2.LINE_AA)

        # Loop through each score and mask for each object we detected
        for score, mask in zip(output['scores'], output['masks']):
            if score > 0.5:

                # Remove extra dimension, move to CPU and convert to numpy array
                current_mask2 = mask.squeeze(0).cpu().numpy()
                # Binary mask that turbns pixels with above 0.5 in 1 and all others as 0
                new_mask = (current_mask2 > 0.5).astype(np.uint8)
                # Find rows and columns where an object is decected, which is when there is a 1
                rows = np.nonzero(new_mask)[0]
                cols = np.nonzero(new_mask)[1]

                # Then calculate centre coords by taking the mean of the rows and columns
                centerX = int(np.mean(cols))
                centerY = int(np.mean(rows))

                # Create placeholder for use in PCACompute, column stack brings the row and columns together as coords for the pixels where there are detected objects
                the_data = np.float32(np.column_stack((cols, rows)))
                mean = cv2.PCACompute(the_data, mean=None)[0]
                # Extract eigenvectors and eigenvalues from the PCACompute functions
                eigenvectors = cv2.PCACompute(the_data, mean=None)[1]
                # Uses PCACompute2 instead of PCACompute as PCACompute doesn't give eigenvalues but PCACompute2 does
                # Eigenvetors and eigenvalues are used in the calculations later to draw principle axes on the detceted objects
                eigenvalues = cv2.PCACompute2(the_data, mean=None)[2]
                
                # Draw circle midpoint at the centre of the detected object
                cv2.circle(result_image4, (centerX, centerY), 5, (0, 0, 0), -1)
                # Calculate length and direction of each principle axis using the eigenvectors and eigenvalues
                # Eigenvalues[0][0] is the variance squared in the direction of the first axis, [1][0] is the variance squared in the direction of the second axis
                length_of_axis1 = np.sqrt(eigenvalues[0][0])
                length_of_axis2 = np.sqrt(eigenvalues[1][0])
                # Eigenvectors give us each coord sets for the direction of the principle axis
                y_direction1 = eigenvectors[0][1]
                x_direction1 = eigenvectors[0][0]
                y_direction2 = eigenvectors[1][1]
                x_direction2 = eigenvectors[1][0]
                # Calculate the end coords for each principle axis with the formula centre plus the axis length multiplied by direction
                x_coord1 = int(centerX + length_of_axis1 * x_direction1)
                y_coord1 = int(centerY + length_of_axis1 * y_direction1)
                x_coord2 = int(centerX + length_of_axis2 * x_direction2)
                y_coord2 = int(centerY + length_of_axis2 * y_direction2)
                # Calculate another set of end cords to go in opposite direction with the formula centre minus the axis length multiplied by direction
                # Which just goes in the other direction but on the same line as the first coords
                x1_coord1 = int(centerX - length_of_axis1 * x_direction1)
                y1_coord1 = int(centerY - length_of_axis1 * y_direction1)
                x1_coord2 = int(centerX - length_of_axis2 * x_direction2)
                y1_coord2 = int(centerY - length_of_axis2 * y_direction2)
                # Draw the lines from the centre to each of the end coords, colour in green with thickness of 2
                cv2.line(result_image4, (centerX, centerY), (x_coord1, y_coord1), (0, 255, 0), 2)
                cv2.line(result_image4, (centerX, centerY), (x1_coord1, y1_coord1), (0, 255, 0), 2)
                cv2.line(result_image4, (centerX, centerY), (x_coord2, y_coord2), (0, 255, 0), 2)
                cv2.line(result_image4, (centerX, centerY), (x1_coord2, y1_coord2), (0, 255, 0), 2)

        # Display the image alongside the masks and bounding boxes
        cv2.imshow("Real-Time (Ish) Mask R-CNN", result_image4)
        # Wait and check to see if q is pressed to quit application
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Stop webcam and close windows
    cap.release()
    cv2.destroyAllWindows()

# Run function
if __name__ == "__main__":
    main()